# Market Basket Analysis
### Data Analyst Internship Project

**Dataset:** UCI Online Retail — real transaction data

This notebook is designed to run end-to-end. It attempts to obtain the official dataset automatically when `data/Online_Retail.csv` is missing. If downloads are blocked, place the official CSV in `data/Online_Retail.csv` and run all cells.

In [ ]:
from pathlib import Path
import io, zipfile, requests, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from mlxtend.frequent_patterns import apriori, association_rules

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
ROOT = Path.cwd()
DATA_DIR = ROOT / "data"
OUT_DIR = ROOT / "outputs"
DATA_DIR.mkdir(exist_ok=True)
OUT_DIR.mkdir(exist_ok=True)
CSV_PATH = DATA_DIR / "Online_Retail.csv"
UCI_URL = "https://archive.ics.uci.edu/static/public/352/online+retail.zip"

In [ ]:
# Automatic dataset preparation: official UCI source only; no synthetic data.
if not CSV_PATH.exists():
    print("Online_Retail.csv not found. Attempting official UCI download...")
    try:
        response = requests.get(UCI_URL, timeout=60)
        response.raise_for_status()
        with zipfile.ZipFile(io.BytesIO(response.content)) as z:
            xlsx_name = next(name for name in z.namelist() if name.lower().endswith(".xlsx"))
            xlsx_bytes = z.read(xlsx_name)
        raw = pd.read_excel(io.BytesIO(xlsx_bytes))
        raw.to_csv(CSV_PATH, index=False)
        print(f"Downloaded and converted {len(raw):,} rows to {CSV_PATH}")
    except Exception as exc:
        raise FileNotFoundError(
            "Automatic download failed. Download the official UCI Online Retail dataset "
            "and place the converted file at data/Online_Retail.csv. Source: " + UCI_URL
        ) from exc
else:
    print(f"Using local dataset: {CSV_PATH}")

## 1. Load and Understand the Dataset

In [ ]:
df = pd.read_csv(CSV_PATH, dtype={"InvoiceNo": str, "StockCode": str, "CustomerID": str})
print("Shape:", df.shape)
display(df.head())
print("\nColumns:")
print(df.columns.tolist())
print("\nData types:")
print(df.dtypes)

In [ ]:
missing = df.isna().sum().sort_values(ascending=False)
duplicates = int(df.duplicated().sum())
print("Missing values:")
display(missing.to_frame("missing_count"))
print("Duplicate rows:", duplicates)

## 2. Data Cleaning

In [ ]:
df["InvoiceNo"] = df["InvoiceNo"].astype(str).str.strip()
df["StockCode"] = df["StockCode"].astype(str).str.strip()
df["Description"] = df["Description"].astype("string").str.strip().str.upper()
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"], errors="coerce")
df["Quantity"] = pd.to_numeric(df["Quantity"], errors="coerce")
df["UnitPrice"] = pd.to_numeric(df["UnitPrice"], errors="coerce")
df["CustomerID"] = pd.to_numeric(df["CustomerID"], errors="coerce")

print("Cancelled invoice lines:", df["InvoiceNo"].str.upper().str.startswith("C").sum())
print("Negative-quantity lines:", (df["Quantity"] < 0).sum())
print("Non-positive price lines:", (df["UnitPrice"] <= 0).sum())

clean_df = df.copy()
clean_df = clean_df[~clean_df["InvoiceNo"].str.upper().str.startswith("C")]
clean_df = clean_df.dropna(subset=["InvoiceNo", "StockCode", "Description", "InvoiceDate", "CustomerID", "Country"])
clean_df = clean_df[(clean_df["Quantity"] > 0) & (clean_df["UnitPrice"] > 0)]
clean_df = clean_df.drop_duplicates()
clean_df["Revenue"] = clean_df["Quantity"] * clean_df["UnitPrice"]
clean_df["InvoiceMonth"] = clean_df["InvoiceDate"].dt.to_period("M").astype(str)
clean_df["InvoiceHour"] = clean_df["InvoiceDate"].dt.hour
print("Clean shape:", clean_df.shape)
display(clean_df.head())

## 3. Exploratory Data Analysis

In [ ]:
top_products = (clean_df.groupby("Description", as_index=False)
                 .agg(Quantity=("Quantity","sum"), Revenue=("Revenue","sum"), Transactions=("InvoiceNo","nunique"))
                 .sort_values("Quantity", ascending=False).head(10))
display(top_products)
plt.figure(figsize=(11,5))
sns.barplot(data=top_products, y="Description", x="Quantity")
plt.title("Top 10 Products by Quantity Sold")
plt.tight_layout(); plt.show()

In [ ]:
country_sales = (clean_df.groupby("Country", as_index=False)
                 .agg(Revenue=("Revenue","sum"), Transactions=("InvoiceNo","nunique"))
                 .sort_values("Revenue", ascending=False).head(10))
display(country_sales)
plt.figure(figsize=(10,5))
sns.barplot(data=country_sales, y="Country", x="Revenue")
plt.title("Top 10 Countries by Revenue")
plt.tight_layout(); plt.show()

In [ ]:
monthly = clean_df.groupby("InvoiceMonth", as_index=False).agg(Revenue=("Revenue","sum"), Transactions=("InvoiceNo","nunique"))
plt.figure(figsize=(12,5))
sns.lineplot(data=monthly, x="InvoiceMonth", y="Revenue", marker="o")
plt.xticks(rotation=45); plt.title("Monthly Revenue Trend"); plt.tight_layout(); plt.show()
display(monthly)

## 4. Basket Size Analysis

In [ ]:
basket_summary = (clean_df.groupby("InvoiceNo")
                   .agg(BasketSize=("StockCode","nunique"), TotalQuantity=("Quantity","sum"), Revenue=("Revenue","sum"))
                   .reset_index())
print(basket_summary.describe())
plt.figure(figsize=(10,5))
sns.histplot(basket_summary["BasketSize"], bins=30, kde=True)
plt.title("Basket Size Distribution"); plt.xlabel("Unique products per transaction"); plt.tight_layout(); plt.show()

## 5. Transaction Matrix and One-Hot Encoding

In [ ]:
# For Market Basket Analysis, each invoice is one basket and each product is a binary feature.
basket = (clean_df.groupby(["InvoiceNo", "Description"])["Quantity"]
          .sum().unstack(fill_value=0))
basket_binary = basket.gt(0).astype(bool)
print("Basket matrix shape:", basket_binary.shape)
display(basket_binary.iloc[:5, :10])

## 6. Apriori — Frequent Itemsets

In [ ]:
# Thresholds are intentionally configurable. Increase min_support if the matrix is large.
MIN_SUPPORT = 0.02
frequent_itemsets = apriori(basket_binary, min_support=MIN_SUPPORT, use_colnames=True, low_memory=True)
frequent_itemsets["item_count"] = frequent_itemsets["itemsets"].apply(len)
frequent_itemsets = frequent_itemsets.sort_values(["support", "item_count"], ascending=[False, True]).reset_index(drop=True)

def format_itemset(s):
    return " | ".join(sorted(map(str, s)))

frequent_itemsets_export = frequent_itemsets.copy()
frequent_itemsets_export["itemsets"] = frequent_itemsets_export["itemsets"].apply(format_itemset)
frequent_itemsets_export.to_csv(OUT_DIR / "frequent_itemsets.csv", index=False)
print("Frequent itemsets:", len(frequent_itemsets_export))
display(frequent_itemsets_export.head(20))

## 7. Association Rules

In [ ]:
MIN_CONFIDENCE = 0.50
rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=MIN_CONFIDENCE)
rules = rules[rules["lift"] >= 1].copy()
rules["antecedents"] = rules["antecedents"].apply(format_itemset)
rules["consequents"] = rules["consequents"].apply(format_itemset)
rules_export = rules[["antecedents","consequents","support","confidence","lift","leverage","conviction"]]
rules_export = rules_export.sort_values(["lift","confidence"], ascending=False).reset_index(drop=True)
rules_export.to_csv(OUT_DIR / "association_rules.csv", index=False)
print("Association rules:", len(rules_export))
display(rules_export.head(20))

In [ ]:
plt.figure(figsize=(9,6))
sns.scatterplot(data=rules_export, x="support", y="confidence", size="lift", hue="lift", palette="viridis", sizes=(40,300), legend=False)
plt.title("Association Rules: Support vs Confidence")
plt.tight_layout(); plt.show()

## 8. Business Insights

In [ ]:
print("BUSINESS INSIGHTS")
print("1. Top-selling products can be prioritized for inventory and merchandising decisions.")
print("2. High-lift rules identify products that co-occur much more often than expected and can support cross-selling or bundle offers.")
print("3. Countries with high revenue represent the strongest existing markets in the observed transaction data.")
print("4. Monthly revenue trends reveal periods that may require additional inventory and operational planning.")
print("5. Basket-size statistics help estimate the typical number of distinct products in an order.")
print("\nTop rules by lift:")
display(rules_export.head(10))

## 9. Power BI Export Check

In [ ]:
print("Created:")
for p in [OUT_DIR / "frequent_itemsets.csv", OUT_DIR / "association_rules.csv"]:
    print(p, "->", p.stat().st_size, "bytes")

## 10. Conclusion and Future Scope

The project demonstrates a complete retail analytics workflow: data quality assessment, cleaning, EDA, basket construction, Apriori frequent-itemset mining, and association-rule generation. The results can support product bundling, cross-selling, merchandising and inventory decisions.

**Future scope:** recommendation systems, country-specific rules, time-aware basket analysis, RFM + association-rule segmentation, and deployment through a dashboard/API.